In [ ]:
import pandas as pd
from collections import Counter
import yaml

In [ ]:
import os
os.chdir('../../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import US patents citation data

In [ ]:
citation_data = pd.read_csv(dataset_config['path_uspat'] + 'g_us_patent_citation.tsv', sep='\t', usecols=['patent_id', 'citation_sequence', 'citation_patent_id'])
citation_data

In [ ]:
location_id = pd.read_csv(dataset_config['path_uspat'] + 'g_assignee_disambiguated.tsv', sep='\t', usecols=['patent_id', 'assignee_sequence', 'location_id'])
location_id = location_id[location_id['assignee_sequence'] == 0]
location_country = pd.read_csv(dataset_config['path_uspat'] + 'g_location_disambiguated.tsv', sep='\t', usecols=['location_id', 'disambig_country'])

In [ ]:
location = pd.merge(location_id, location_country)
location = location.drop(columns=['assignee_sequence', 'location_id'])
location.rename(
    columns={'patent_id': 'citation_patent_id',
             'disambig_country': 'country'},
    inplace=True
)
location

### Merge citation country

In [ ]:
citation_country = pd.merge(citation_data, location, on='citation_patent_id')
citation_country

In [ ]:
# Check
subset = citation_country[citation_country['patent_id'] == 3978580] 
print(subset)


## Calculate

In [ ]:
# For each patent_id, calculate the num and share of citations from the US
pct_us = (
    citation_country
    .assign(is_us = (citation_country['country'] == 'US').astype(int))
    .groupby('patent_id', as_index=False)
    .agg(
        num_us_citations = ('is_us', 'sum'),
        pct_us_citations = ('is_us', 'mean')
    )
)

pct_us

In [ ]:
pct_us.to_csv(dataset_config['path_processed'] + 'CN_CN/USpat_citepat_country.csv', index=False)